# Demo: substitucni sifra a kryptoanalyza

Notebook ukazuje zakladni workflow podle zadani: sifrovani, desifrovani, referencni bigramovou matici, kratkou ukazku Metropolis-Hastings algoritmu a export vysledku. Pro finalni desifrovani zadanych souboru se podle PDF pouziva 20 000 iteraci na text.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from substitution_cipher import (
    ALPHABET,
    build_reference_matrix_from_text,
    export_result,
    get_bigrams,
    plausibility,
    prolom_substitute,
    substitute_decrypt,
    substitute_encrypt,
)
from substitution_cipher.bigrams import load_matrix, save_matrix

ALPHABET, len(ALPHABET)

## Sifrovani a desifrovani

In [ ]:
plaintext = 'BYL_POZDNI_VECER_PRVNI_MAJ'
key = ALPHABET[3:] + ALPHABET[:3]
ciphertext = substitute_encrypt(plaintext, key)
decrypted = substitute_decrypt(ciphertext, key)

plaintext, key, ciphertext, decrypted

## Referencni bigramova matice

In [ ]:
clean_text_path = PROJECT_ROOT / 'data' / 'processed' / 'clean_text.txt'
matrix_path = PROJECT_ROOT / 'data' / 'processed' / 'TM_ref.npy'

clean_text = clean_text_path.read_text(encoding='utf-8').strip()
bigrams = get_bigrams(clean_text)

if matrix_path.exists():
    TM_ref = load_matrix(matrix_path)
else:
    TM_ref = build_reference_matrix_from_text(clean_text)
    save_matrix(TM_ref, matrix_path)

len(clean_text), len(bigrams), TM_ref.shape, float(TM_ref.sum()), float(TM_ref.min())

## Vizualizace matice

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 7))
plt.imshow(TM_ref, cmap='viridis')
plt.xticks(range(len(ALPHABET)), list(ALPHABET), rotation=90)
plt.yticks(range(len(ALPHABET)), list(ALPHABET))
plt.colorbar(label='pravdepodobnost')
plt.title('Referencni bigramova matice')
plt.tight_layout()
plt.show()

## Kratka ukazka kryptoanalyzy

Tato ukazka pouziva malo iteraci, aby byl notebook rychly. Pro finalni soubory od vyucujiciho spousti pripraveny skript `scripts/decrypt_samples.py` implicitne 20 000 iteraci.

In [ ]:
demo_plaintext = 'AHOJ_SVETE_AHOJ_SVETE'
demo_key = ALPHABET[5:] + ALPHABET[:5]
demo_ciphertext = substitute_encrypt(demo_plaintext, demo_key)

best_key, best_text, best_score = prolom_substitute(
    demo_ciphertext,
    TM_ref,
    iter=200,
    seed=42,
    progress_every=0,
)

demo_ciphertext, best_key, best_text, best_score

## Export vysledku

In [ ]:
plaintext_path, key_path = export_result(
    plaintext=best_text,
    key=best_key,
    text_length=len(demo_ciphertext),
    sample_id=1,
    output_dir=PROJECT_ROOT / 'outputs' / 'demo',
)

plaintext_path.name, key_path.name

## Vyhodnoceni

Referencni matice ma tvar 27 x 27, je vyhlazena proti nulam a je normalizovana na soucet 1. Metropolis-Hastings prohledava prostor klicu pres nahodne prohozeni dvou znaku v klici. U realnych zadanych ciphertextu je vhodne spoustet vice behu a pouzit alespon 20 000 iteraci na text, jak pozaduje PDF zadani.